# Z-averages via PyTecplot (IMax-robust)

Connects to a **running Tecplot 360 session** and executes the z-average equations directly — no more copy-pasting generated strings.

**Setup (once):**
1. In Tecplot 360: `Scripting > PyTecplot Connections... > Accept connections` (default port 7600)
2. `pip install pytecplot`

**What it fixes:** zones are selected by *inspecting the dataset* (zone count, IMax, variable presence), so files with truncated stations / different IMax just work.

In [1]:
import tecplot as tp

tp.session.connect()  # port=7600 by default; tp.session.connect(port=7601) if you changed it
ds = tp.active_frame().dataset
print(ds)
print(f"zones: {ds.num_zones}, variables: {ds.num_variables}")

Connecting to Tecplot 360 TecUtil Server on:
    tcp://localhost:7600
Connection established.
Dataset: 'Fluent Common Fluid Files Data | Fluent Common Fluid Files Data'
  Zones: 'Time Average - 11', 'Time Average - 12', 'Time Average - 13',
    'LES c xh2', 'Slice: Z=-2.94059', 'Slice: Z=-2.88119', 'Slice: Z=-2.82178',
    'Slice: Z=-2.76238', 'Slice: Z=-2.70297', 'Slice: Z=-2.64356',
    'Slice: Z=-2.58416', 'Slice: Z=-2.52475', 'Slice: Z=-2.46535',
    'Slice: Z=-2.40594', 'Slice: Z=-2.34653', 'Slice: Z=-2.28713',
    'Slice: Z=-2.22772', 'Slice: Z=-2.16832', 'Slice: Z=-2.10891',
    'Slice: Z=-2.0495', 'Slice: Z=-1.9901', 'Slice: Z=-1.93069',
    'Slice: Z=-1.87129', 'Slice: Z=-1.81188', 'Slice: Z=-1.75248',
    'Slice: Z=-1.69307', 'Slice: Z=-1.63366', 'Slice: Z=-1.57426',
    'Slice: Z=-1.51485', 'Slice: Z=-1.45545', 'Slice: Z=-1.39604',
    'Slice: Z=-1.33663', 'Slice: Z=-1.27723', 'Slice: Z=-1.21782',
    'Slice: Z=-1.15842', 'Slice: Z=-1.09901', 'Slice: Z=-1.0396',
    'Slice: 

## 1. Inspect the dataset

First look at what's actually there: zone index (1-based), name, zone type, and point count. The z-slice zones are FE-line zones (no `IMax`), so we use `num_points` as the point-count stand-in — this is informational only now, since section 4 no longer requires matching point counts across zones (it interpolates by position instead).

In [2]:
import pandas as pd

rows = []
for z in ds.zones():
    try:
        imax, jmax, kmax = z.dimensions
    except AttributeError:
        # FE zones (e.g. the "Slice: Z=..." FE-line zones) have no IJK dimensions;
        # num_points is the closest equivalent and is what varies between slices.
        imax, jmax, kmax = z.num_points, None, None
    rows.append({
        'eq_index': z.index + 1,  # 1-based index used in {var}[i] equations
        'name': z.name,
        'zone_type': str(z.zone_type),
        'IMax': imax, 'JMax': jmax, 'KMax': kmax,
    })
zones_df = pd.DataFrame(rows)

print(zones_df['IMax'].value_counts().rename_axis('IMax / NumPoints').rename('n_zones'))
zones_df

IMax / NumPoints
242      234
244       53
246       15
250        3
248        2
252        2
11860      1
11945      1
11858      1
Name: n_zones, dtype: int64


,eq_index,name,zone_type,IMax,JMax,KMax
0,1,Time Average - 11,ZoneType.FEPolygon,11860,NaN,NaN
1,2,Time Average - 12,ZoneType.FEPolygon,11945,NaN,NaN
2,3,Time Average - 13,ZoneType.FEPolygon,11858,NaN,NaN
3,4,LES c xh2,ZoneType.Ordered,242,1.0,1.0
4,5,Slice: Z=-2.94059,ZoneType.Ordered,242,1.0,1.0
...,...,...,...,...,...,...
307,308,Slice: Z=2.76238,ZoneType.Ordered,242,1.0,1.0
308,309,Slice: Z=2.82178,ZoneType.Ordered,242,1.0,1.0
309,310,Slice: Z=2.88119,ZoneType.Ordered,242,1.0,1.0
310,311,Slice: Z=2.94059,ZoneType.Ordered,242,1.0,1.0


## 1b. Set interpolation position variables

`tp.data.operate.interpolate_linear` / `interpolate_inverse_distance` match points by the frame's assigned X/Y(/Z) axis variables, not by array index. Each z-slice varies in `y/h` at a fixed `z/h`; `z/h` differs slice-to-slice and is exactly what we're averaging over, so it must **not** be part of the position match — only `x/h, y/h` should be. Forcing 2D Cartesian here is what actually fixes the "different point counts" problem: it makes matching based on where a point sits, not on how many points came before it.

In [3]:
frame = tp.active_frame()
frame.plot_type = tp.constant.PlotType.Cartesian2D
plot = frame.plot()
plot.axes.x_axis.variable = ds.variable('x/h')
plot.axes.y_axis.variable = ds.variable('y/h')
print(f"interpolation position variables: X={plot.axes.x_axis.variable.name}, Y={plot.axes.y_axis.variable.name}")

interpolation position variables: X=x/h, Y=y/h


In [4]:
# Variables present (equations fail if a variable doesn't exist)
print([v.name for v in ds.variables()])

['x/h', 'y/h', 'z/h', 'theta', 'normalized_u', 'u_rms', 'v_rms', 'theta_rms', 'u_theta', 'v_theta', 'uv', 'k_res', 'k', 'u_avg_z', 'k_t_avg_z', 'theta_avg_z', 'theta_rms_avg_z', 'u_theta_avg_z', 'v_theta_avg_z', 'uv_avg_z', '_zavg_src_uv', '_zavg_src_v_theta', '_zavg_src_normalized_u', '_zavg_src_theta_rms', '_zavg_src_u_theta', '_zavg_src_k', '_zavg_src_theta']


## 2. Configuration

Stations are auto-detected from the `LES c xh*` zone names (see section 3) — no manual index bookkeeping needed. Only this cell should need editing between cases.

In [5]:
# output variable -> source variable
VARIABLES = {
    'u_avg_z':         'normalized_u',
    'k_t_avg_z':       'k',
    'theta_avg_z':     'theta',
    'theta_rms_avg_z': 'theta_rms',
    'u_theta_avg_z':   'u_theta',
    'v_theta_avg_z':   'v_theta',
    'uv_avg_z':        'uv',
}

SKIP_INDICES = set()          # e.g. {58, 59, 622} — 1-based eq_index of slices to exclude
EXCLUDE_MIDPOINT = True       # drop the 'Slice: Z=0' zone from each station's average

# 'linear'            -> tp.data.operate.interpolate_linear (requires the destination point
#                        to fall *inside* an enclosing source element; fails here because the
#                        slices are line segments with ~zero area in the x/h-y/h plane)
# 'inverse_distance'  -> tp.data.operate.interpolate_inverse_distance (no enclosing-element
#                        requirement — works for line/point data like these slices)
INTERP_METHOD = 'inverse_distance'
IDW_NUM_POINTS = 4

## 3. Auto-detect stations and build the zone filter

Every zone named `LES c xh*` is a station's destination zone; the slices belonging to it are all zones between it and the next `LES c xh*` zone (or the end of the dataset).

In [6]:
# matches "LES c xh2", "LES f xh4", "RSM xh10", etc. — any name containing "xh<number>"
les_zones = zones_df[zones_df['name'].str.contains(r'xh\d', regex=True)].sort_values('eq_index').reset_index(drop=True)
print(les_zones[['eq_index', 'name']])

last_index = int(zones_df['eq_index'].max())

stations = {}
for pos, row in les_zones.iterrows():
    label = row['name']
    dest_idx = int(row['eq_index'])
    next_idx = int(les_zones['eq_index'].iloc[pos + 1]) if pos + 1 < len(les_zones) else last_index + 1
    stations[label] = {'dest': dest_idx, 'slice_range': (dest_idx + 1, next_idx - 1)}

name_by_index = dict(zip(zones_df['eq_index'], zones_df['name']))

def station_zones(info):
    start, end = info['slice_range']
    zones, dropped = [], {'midpoint': [], 'skip': [], 'missing': []}
    for i in range(start, end + 1):
        if i not in name_by_index:
            dropped['missing'].append(i)
        elif EXCLUDE_MIDPOINT and name_by_index[i] == 'Slice: Z=0':
            dropped['midpoint'].append(i)
        elif i in SKIP_INDICES:
            dropped['skip'].append(i)
        else:
            zones.append(i)
    return zones, dropped

# dry run: show what each station will use, BEFORE touching the dataset
plan = {}
for label, info in stations.items():
    zones, dropped = station_zones(info)
    plan[label] = zones
    start, end = info['slice_range']
    print(f"{label}: dest zone {info['dest']}, slices {start}..{end} -> averaging {len(zones)} zones")
    for reason, idxs in dropped.items():
        if idxs:
            shown = idxs if len(idxs) <= 8 else idxs[:4] + ['...'] + idxs[-2:]
            print(f"    dropped ({reason}): {shown}")

   eq_index        name
0         4   LES c xh2
1       107   LES c xh4
2       210  LES c xh10
LES c xh2: dest zone 4, slices 5..106 -> averaging 101 zones
    dropped (midpoint): [55]
LES c xh4: dest zone 107, slices 108..209 -> averaging 101 zones
    dropped (midpoint): [158]
LES c xh10: dest zone 210, slices 211..312 -> averaging 101 zones
    dropped (midpoint): [261]


## 4. Execute the averages

For each slice, its source data is copied into a scratch variable, spatially interpolated onto the destination zone's own points (via `interpolate_linear`/`interpolate_inverse_distance`, matching on `x/h, y/h` per section 1b), then accumulated into the destination's `*_avg_z` variable. Because matching is by position rather than by array index, slices no longer need matching point counts.

In [7]:
var_names = {v.name for v in ds.variables()}
existing_var_names = set(var_names)

# one scratch variable per distinct source variable, reused across every slice/station
scratch_names = {src: f"_zavg_src_{src}" for src in set(VARIABLES.values())}
for scratch in scratch_names.values():
    if scratch not in existing_var_names:
        ds.add_variable(scratch)
        existing_var_names.add(scratch)

def interpolate(dest_zone, src_zone, variables):
    if INTERP_METHOD == 'linear':
        tp.data.operate.interpolate_linear(destination_zone=dest_zone, source_zones=[src_zone], variables=variables)
    else:
        tp.data.operate.interpolate_inverse_distance(
            destination_zone=dest_zone, source_zones=[src_zone], variables=variables,
            point_selection=tp.constant.PtSelection.NearestN, num_points=IDW_NUM_POINTS,
        )

for label, info in stations.items():
    zones = plan[label]
    dest_zone = ds.zone(info['dest'] - 1)
    if not zones:
        print(f"{label}: no valid zones, skipping")
        continue

    active_vars = {out: src for out, src in VARIABLES.items() if src in var_names}
    missing = [src for src in VARIABLES.values() if src not in var_names]
    if missing:
        print(f"    !! variables not in dataset, skipping: {missing}")

    n = len(zones)
    print(f"{label}: averaging {n} zones -> {dest_zone.name} (zone {info['dest']})")

    for out_var in active_vars:
        tp.data.operate.execute_equation(f"{{{out_var}}} = 0", zones=[dest_zone])

    scratch_vars = [ds.variable(scratch_names[src]) for src in set(active_vars.values())]

    for i in zones:
        src_zone = ds.zone(i - 1)

        for src in set(active_vars.values()):
            tp.data.operate.execute_equation(f"{{{scratch_names[src]}}} = {{{src}}}", zones=[src_zone])

        interpolate(dest_zone, src_zone, scratch_vars)

        for out_var, src in active_vars.items():
            tp.data.operate.execute_equation(
                f"{{{out_var}}} = {{{out_var}}} + {{{scratch_names[src]}}}/{n}", zones=[dest_zone]
            )

    print(f"    ok: {list(active_vars)} averaged over {n} zones")

print('\ndone')

LES c xh2: averaging 101 zones -> LES c xh2 (zone 4)
    ok: ['u_avg_z', 'k_t_avg_z', 'theta_avg_z', 'theta_rms_avg_z', 'u_theta_avg_z', 'v_theta_avg_z', 'uv_avg_z'] averaged over 101 zones
LES c xh4: averaging 101 zones -> LES c xh4 (zone 107)
    ok: ['u_avg_z', 'k_t_avg_z', 'theta_avg_z', 'theta_rms_avg_z', 'u_theta_avg_z', 'v_theta_avg_z', 'uv_avg_z'] averaged over 101 zones
LES c xh10: averaging 101 zones -> LES c xh10 (zone 210)
    ok: ['u_avg_z', 'k_t_avg_z', 'theta_avg_z', 'theta_rms_avg_z', 'u_theta_avg_z', 'v_theta_avg_z', 'uv_avg_z'] averaged over 101 zones

done


## 5. (Optional) sanity check a result

Since the average is now built through spatial interpolation rather than exact index matching, an exact NumPy replica isn't meaningful. Instead: for one destination point, pull the nearest-`y/h` source value from each slice and compare its plain mean to what Tecplot computed — they should be close (not identical, since Tecplot did real interpolation, this does nearest-point).

In [8]:
import numpy as np

label = next(iter(stations))
info = stations[label]
zones = plan[label]
dest_zone = ds.zone(info['dest'] - 1)
out_var, src_var = 'u_avg_z', VARIABLES['u_avg_z']

dest_y = np.array(dest_zone.values('y/h')[:])
dest_result = np.array(dest_zone.values(out_var)[:])

point_idx = len(dest_y) // 2
y_target = dest_y[point_idx]

nearest_vals = []
for i in zones:
    src_zone = ds.zone(i - 1)
    y = np.array(src_zone.values('y/h')[:])
    v = np.array(src_zone.values(src_var)[:])
    nearest_vals.append(v[np.argmin(np.abs(y - y_target))])

manual_mean = np.mean(nearest_vals)
tec_value = dest_result[point_idx]

print(f"{label}: at y/h={y_target:.4f} (point {point_idx})")
print(f"  Tecplot ({INTERP_METHOD} interp) {out_var} = {tec_value:.6g}")
print(f"  nearest-point manual mean over {len(zones)} slices = {manual_mean:.6g}")
print(f"  relative difference = {abs(tec_value - manual_mean) / (abs(manual_mean) + 1e-30):.2%}")

LES c xh2: at y/h=-0.1250 (point 121)
  Tecplot (inverse_distance interp) u_avg_z = 1.10377
  nearest-point manual mean over 101 slices = 1.10377
  relative difference = 0.00%
